In [81]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error

#created toy dataset for 5 users across 10 movies to make calculations by hand easy
ratings_matrix = pd.DataFrame({
    "Movie1": [5, 4, np.nan, 3, 5],
    "Movie2": [4, np.nan, 5, 3, np.nan],
    "Movie3": [np.nan, 5, np.nan, 4, np.nan],
    "Movie4": [5, 4, 5, np.nan, np.nan],
    "Movie5": [3, np.nan, 5, 2, np.nan],
    "Movie6": [4, 3, np.nan, 2, 4],
    "Movie7": [4, 3, 2, np.nan, 4],
    "Movie8": [4, np.nan, 4, np.nan, 2],
    "Movie9": [5, np.nan, 3, 3, 4],
    "Movie10": [np.nan, 3, 5, np.nan, 4]
}, index=["User1", "User2", "User3", "User4", "User5"])

#pivot long so movies and ratings have their own columns
ratings_long = ratings_matrix.reset_index().melt(
    id_vars="index",
    var_name="movie",
    value_name="rating"
).rename(columns={"index": "user"})

ratings_long = ratings_long.dropna()

print(ratings_matrix)
print(ratings_long)



       Movie1  Movie2  Movie3  Movie4  Movie5  Movie6  Movie7  Movie8  Movie9  \
User1     5.0     4.0     NaN     5.0     3.0     4.0     4.0     4.0     5.0   
User2     4.0     NaN     5.0     4.0     NaN     3.0     3.0     NaN     NaN   
User3     NaN     5.0     NaN     5.0     5.0     NaN     2.0     4.0     3.0   
User4     3.0     3.0     4.0     NaN     2.0     2.0     NaN     NaN     3.0   
User5     5.0     NaN     NaN     NaN     NaN     4.0     4.0     2.0     4.0   

       Movie10  
User1      NaN  
User2      3.0  
User3      5.0  
User4      NaN  
User5      4.0  
     user    movie  rating
0   User1   Movie1     5.0
1   User2   Movie1     4.0
3   User4   Movie1     3.0
4   User5   Movie1     5.0
5   User1   Movie2     4.0
7   User3   Movie2     5.0
8   User4   Movie2     3.0
11  User2   Movie3     5.0
13  User4   Movie3     4.0
15  User1   Movie4     5.0
16  User2   Movie4     4.0
17  User3   Movie4     5.0
20  User1   Movie5     3.0
22  User3   Movie5     5.0
23  Us

In [82]:
#split test/training data 

train, test = train_test_split(
    ratings_long,
    test_size=0.2,
    random_state=42
)

#create global mean and raw mean for test/train
global_mean = train["rating"].mean()

train["mean_pred"] = global_mean
test["mean_pred"] = global_mean

#calculate RMSE for test/training dataset
raw_train_rmse = root_mean_squared_error(train["rating"],train["mean_pred"])

raw_test_rmse = root_mean_squared_error(test["rating"],test["mean_pred"])

print("global mean:", global_mean)
print("train:", raw_train_rmse, "test:", raw_test_rmse)


global mean: 3.6923076923076925
train: 0.9514859136040755 test: 1.0874679968280012


In [83]:
#extract user and movie bias from dataset
#user bias = user mean - global_mean
#movie bias = movie mean - global_mean
user_bias = train.groupby("user")["rating"].mean() - global_mean
movie_bias = train.groupby("movie")["rating"].mean() - global_mean

#add user and movie bias back into datasets
train["user_bias"] = train["user"].map(user_bias).fillna(0)
train["movie_bias"] = train["movie"].map(movie_bias).fillna(0)

test["user_bias"] = test["user"].map(user_bias).fillna(0)
test["movie_bias"] = test["movie"].map(movie_bias).fillna(0)

print("user bias:", user_bias, "movie_bias:", movie_bias)

user bias: user
User1    0.307692
User2   -0.025641
User3    0.307692
User4   -0.942308
User5    0.141026
Name: rating, dtype: float64 movie_bias: movie
Movie1     0.557692
Movie10   -0.192308
Movie2     0.307692
Movie3     1.307692
Movie4     0.807692
Movie5    -0.358974
Movie6    -0.192308
Movie7    -0.692308
Movie8    -0.358974
Movie9    -0.358974
Name: rating, dtype: float64


In [84]:
#baseline predictions added into datasets

train["baseline_pred"] = (global_mean + train["user_bias"] + train["movie_bias"])

test["baseline_pred"] = (global_mean + test["user_bias"] + test["movie_bias"])

print(train)

print(test)


     user    movie  rating  mean_pred  user_bias  movie_bias  baseline_pred
32  User3   Movie7     2.0   3.692308   0.307692   -0.692308       3.307692
20  User1   Movie5     3.0   3.692308   0.307692   -0.358974       3.641026
0   User1   Movie1     5.0   3.692308   0.307692    0.557692       4.557692
5   User1   Movie2     4.0   3.692308   0.307692    0.307692       4.307692
26  User2   Movie6     3.0   3.692308  -0.025641   -0.192308       3.474359
7   User3   Movie2     5.0   3.692308   0.307692    0.307692       4.307692
22  User3   Movie5     5.0   3.692308   0.307692   -0.358974       3.641026
17  User3   Movie4     5.0   3.692308   0.307692    0.807692       4.807692
37  User3   Movie8     4.0   3.692308   0.307692   -0.358974       3.641026
1   User2   Movie1     4.0   3.692308  -0.025641    0.557692       4.224359
3   User4   Movie1     3.0   3.692308  -0.942308    0.557692       3.307692
44  User5   Movie9     4.0   3.692308   0.141026   -0.358974       3.474359
4   User5   

In [86]:
#baseline rmse for both

baseline_train_rmse = root_mean_squared_error(train["rating"],train["baseline_pred"])

baseline_test_rmse = root_mean_squared_error(test["rating"],test["baseline_pred"])

print("Baseline train RMSE:", baseline_train_rmse)
print("Baseline test RMSE:", baseline_test_rmse)

results = pd.DataFrame({
    "Model Type": ["Raw Average", "Baseline Bias"],
    "Training RMSE": [raw_train_rmse, baseline_train_rmse],
    "Test RMSE": [raw_test_rmse, baseline_test_rmse]
})

print(results)

Baseline train RMSE: 0.6387459463338475
Baseline test RMSE: 0.7686734855563766
      Model Type  Training RMSE  Test RMSE
0    Raw Average       0.951486   1.087468
1  Baseline Bias       0.638746   0.768673


# Summary
Model Type   Training RMSE   Test RMSE
Raw Average       0.951486   1.087468
Baseline Bias       0.638746   0.768673


The raw average model predicted every user movie rating using only the mean rating from the training data. The training rmse is 0.9515 and test rmse is 1.0875. The baseline bias improved the prediction by adding user bias and movie bias to the global average. This was able to reduce the training rmse to 0.6387 and the test rmse to 0.7687. The baseline model did have a lower rmse on both the training and test datasets and it performed better than the raw average model. Our improvement from a percentage was about 33% for the training rmse and 29% for the test rmse. From a business perspective in a recommender this would represent a very substantial improvement that could become critical in user retention. This is also a verification of the hypothesis that by accounting for user rating tendencies and movie popularity we were able to improve prediction accuracy. 